In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

In [8]:
ROOT_DIR = Path.cwd()
SUMMARY_PATH = ROOT_DIR / "results" / "training_summary.csv"
METRIC = "weighted_accuracy_score"
CLASS_LABELS = {0: "CN", 1: "MCI", 2: "AD"}
VIEW_LABELS = {1: "Clinical", 2: "MRI", 3: "PET"}

df = pd.read_csv(SUMMARY_PATH).copy()
df.head()

,dataset,epochs,view_1,view_2,view_3,class_0,class_1,class_2,hidden_dim_1,hidden_dim_2,hidden_dim_3,avg_uncertainty,weighted_accuracy_score,auc_score,macro_f1_score,model_save_path
0,aibl,644,1,0,0,1,1,0,128,64,64,0.088940,0.850000,0.578467,0.549625,models//20260424_164805.pth
1,aibl,68,1,0,0,1,0,1,128,64,64,0.372535,0.795000,0.755189,0.603462,models//20260424_165041.pth
2,aibl,1,1,0,0,0,1,1,128,64,64,0.535964,0.553191,0.585185,0.553191,models//20260424_165226.pth
3,aibl,52,0,1,0,1,1,0,128,64,64,0.626296,0.888889,0.639712,0.553571,models//20260424_165300.pth


In [9]:
def build_class_pair(row):
    active = [CLASS_LABELS[idx] for idx in CLASS_LABELS if row[f"class_{idx}"] == 1]
    return " vs ".join(active)


def build_view_ids(row):
    return tuple(idx for idx in VIEW_LABELS if row[f"view_{idx}"] == 1)


def build_view_name(view_ids):
    return " + ".join(VIEW_LABELS[idx] for idx in view_ids)


df["class_pair"] = df.apply(build_class_pair, axis=1)
df["view_ids"] = df.apply(build_view_ids, axis=1)
df["view_count"] = df["view_ids"].str.len()
df["view_name"] = df["view_ids"].apply(build_view_name)
df["hidden_dim"] = df[["hidden_dim_1", "hidden_dim_2", "hidden_dim_3"]].astype(str).agg("/".join, axis=1)

rank_columns = [METRIC, "macro_f1_score", "auc_score", "avg_uncertainty"]
ascending = [False, False, False, True]

summary_cols = [
    "dataset",
    "class_pair",
    "view_name",
    METRIC,
    "macro_f1_score",
    "auc_score",
    "avg_uncertainty",
    "epochs",
    "hidden_dim",
    "model_save_path",
]

df[summary_cols].sort_values(["dataset", "class_pair", METRIC], ascending=[True, True, False]).head(10)

,dataset,class_pair,view_name,weighted_accuracy_score,macro_f1_score,auc_score,avg_uncertainty,epochs,hidden_dim,model_save_path
1,aibl,CN vs AD,Clinical,0.795000,0.603462,0.755189,0.372535,68,128/64/64,models//20260424_165041.pth
3,aibl,CN vs MCI,MRI,0.888889,0.553571,0.639712,0.626296,52,128/64/64,models//20260424_165300.pth
0,aibl,CN vs MCI,Clinical,0.850000,0.549625,0.578467,0.088940,644,128/64/64,models//20260424_164805.pth
2,aibl,MCI vs AD,Clinical,0.553191,0.553191,0.585185,0.535964,1,128/64/64,models//20260424_165226.pth


In [10]:
single_view_df = df[df["view_count"] == 1].copy()
single_view_best = (
    single_view_df
    .sort_values(["dataset", "class_pair"] + rank_columns, ascending=[True, True] + ascending)
    .groupby(["dataset", "class_pair"], as_index=False)
    .first()
)
single_view_best["best_modality_id"] = single_view_best["view_ids"].str[0]
single_view_best["best_modality"] = single_view_best["best_modality_id"].map(VIEW_LABELS)

best_single_view_summary = single_view_best[
    [
        "dataset",
        "class_pair",
        "best_modality",
        "view_name",
        METRIC,
        "macro_f1_score",
        "auc_score",
        "avg_uncertainty",
        "epochs",
        "hidden_dim",
        "model_save_path",
    ]
].rename(columns={"view_name": "best_single_view"})

best_single_view_summary

,dataset,class_pair,best_modality,best_single_view,weighted_accuracy_score,macro_f1_score,auc_score,avg_uncertainty,epochs,hidden_dim,model_save_path
0,aibl,CN vs AD,Clinical,Clinical,0.795000,0.603462,0.755189,0.372535,68,128/64/64,models//20260424_165041.pth
1,aibl,CN vs MCI,MRI,MRI,0.888889,0.553571,0.639712,0.626296,52,128/64/64,models//20260424_165300.pth
2,aibl,MCI vs AD,Clinical,Clinical,0.553191,0.553191,0.585185,0.535964,1,128/64/64,models//20260424_165226.pth


In [11]:
two_view_df = df[df["view_count"] == 2].copy()
best_pairs = []

for _, single_row in single_view_best.iterrows():
    candidates = two_view_df[
        (two_view_df["dataset"] == single_row["dataset"])
        & (two_view_df["class_pair"] == single_row["class_pair"])
        & (two_view_df["view_ids"].apply(lambda ids: single_row["best_modality_id"] in ids))
    ].copy()

    if candidates.empty:
        continue

    best_candidate = candidates.sort_values(rank_columns, ascending=ascending).iloc[0]
    record = single_row.to_dict()
    record.update({
        "best_two_view": best_candidate["view_name"],
        f"best_two_view_{METRIC}": best_candidate[METRIC],
        "best_two_view_macro_f1_score": best_candidate["macro_f1_score"],
        "best_two_view_auc_score": best_candidate["auc_score"],
        "best_two_view_avg_uncertainty": best_candidate["avg_uncertainty"],
        "best_two_view_epochs": best_candidate["epochs"],
        "best_two_view_hidden_dim": best_candidate["hidden_dim"],
        "best_two_view_model_save_path": best_candidate["model_save_path"],
    })
    best_pairs.append(record)

best_two_view_summary = pd.DataFrame(best_pairs)[
    [
        "dataset",
        "class_pair",
        "best_modality",
        "view_name",
        METRIC,
        "best_two_view",
        f"best_two_view_{METRIC}",
        "macro_f1_score",
        "best_two_view_macro_f1_score",
        "auc_score",
        "best_two_view_auc_score",
        "avg_uncertainty",
        "best_two_view_avg_uncertainty",
    ]
].rename(columns={
    "view_name": "best_single_view",
    METRIC: f"best_single_view_{METRIC}",
    "macro_f1_score": "best_single_view_macro_f1_score",
    "auc_score": "best_single_view_auc_score",
    "avg_uncertainty": "best_single_view_avg_uncertainty",
})

best_two_view_summary

KeyError: "None of [Index(['dataset', 'class_pair', 'best_modality', 'view_name',\n       'weighted_accuracy_score', 'best_two_view',\n       'best_two_view_weighted_accuracy_score', 'macro_f1_score',\n       'best_two_view_macro_f1_score', 'auc_score', 'best_two_view_auc_score',\n       'avg_uncertainty', 'best_two_view_avg_uncertainty'],\n      dtype='object')] are in the [columns]"

In [ ]:
output_map = {
    "CN vs MCI": ROOT_DIR / "cn_mci_results.csv",
    "CN vs AD": ROOT_DIR / "cn_ad_results.csv",
    "MCI vs AD": ROOT_DIR / "mci_ad_results.csv",
}

for class_pair, output_path in output_map.items():
    class_df = best_two_view_summary[best_two_view_summary["class_pair"] == class_pair].copy()
    if not class_df.empty:
        class_df.to_csv(output_path, index=False)

best_two_view_summary